# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [ ]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

In [ ]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

# Memory Monitoring Utilities

In [ ]:
def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def format_bytes(bytes):
    """Format bytes to human readable format"""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes < 1024.0:
            return f"{bytes:.2f} {unit}"
        bytes /= 1024.0
    return f"{bytes:.2f} PB"

def estimate_chunk_memory(width, height, bands, dtype):
    """Estimate memory requirement for a chunk"""
    dtype_sizes = {
        'uint8': 1, 'uint16': 2, 'uint32': 4,
        'int8': 1, 'int16': 2, 'int32': 4,
        'float32': 4, 'float64': 8
    }
    bytes_per_pixel = dtype_sizes.get(str(dtype), 4)
    return width * height * bands * bytes_per_pixel

def calculate_optimal_chunk_size(raster_width, raster_height, bands, dtype, memory_limit_mb=500):
    """Calculate optimal chunk size based on available memory"""
    memory_limit_bytes = memory_limit_mb * 1024 * 1024
    
    # Start with default chunk size
    chunk_size = 1024
    
    # Calculate memory for default chunk
    chunk_memory = estimate_chunk_memory(chunk_size, chunk_size, bands, dtype)
    
    # Adjust chunk size if needed
    if chunk_memory > memory_limit_bytes:
        # Calculate maximum chunk size that fits in memory
        bytes_per_pixel = chunk_memory / (chunk_size * chunk_size)
        max_pixels = memory_limit_bytes / bytes_per_pixel
        chunk_size = int(np.sqrt(max_pixels))
        # Round down to nearest power of 2 for efficiency
        chunk_size = 2 ** int(np.log2(chunk_size))
    
    # Ensure chunk size is at least 256
    chunk_size = max(256, chunk_size)
    
    print(f"📊 Optimal chunk size: {chunk_size}x{chunk_size}")
    print(f"   Estimated memory per chunk: {format_bytes(estimate_chunk_memory(chunk_size, chunk_size, bands, dtype))}")
    
    return chunk_size

print("✅ Memory monitoring utilities loaded")

# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [ ]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [ ]:
EVENT_NAME = '202405_Flood_TX'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'sentinel2'      #find the name within drcs_activations OLD Directory (see link above)

RENAME_PRODUCT = 'Sentinel-2'   #choose from LIST of 2nd level directories (see above list)

PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory
DIRECTORY_NEW = f'{DIR_NEW_BASE}/{RENAME_PRODUCT}'

In [ ]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [ ]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

# For these we can see three different types of files

1. WM = water mask
2. rgb = red green blue
3. WM_diff = water mask difference between dates

### We are going to need 2 different directories for these!!!

We will keep WaterMask (WM) and rgb as separate directories

In [ ]:
# For simplicity, let's use python list comprehension to return the files
# We may need to rename them in different ways for different products
# We will do a similar process later

## NOTE --- We can actually use these objects since they have the same path as the s3 files. We will call them again later

cir = [f for f in keys if "cir" in f]
print(cir)
swir = [f for f in keys if "swir" in f]
print(swir)
true = [f for f in keys if "true" in f]
print(true)

In [ ]:
config_cir = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/cir",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

config_swir = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/swir", #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

config_true = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/true", #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

## Configure bucket and paths (no need to create session manually)

In [ ]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [ ]:
# Use the function with config_WM
cir_bucket = return_bucket_info(config_cir)
true_bucket = return_bucket_info(config_true)
swir_bucket = return_bucket_info(config_swir)

In [ ]:
def convert_sentinel_datetime(datetime_str):
    """
    Convert Sentinel datetime format to ISO 8601 format with UTC timezone.
    
    Args:
        datetime_str: String like '20240430T002653'
    
    Returns:
        String like '2024-04-30T00:26:53Z'
    """
    # Extract components
    year = datetime_str[0:4]
    month = datetime_str[4:6]
    day = datetime_str[6:8]
    hour = datetime_str[9:11]
    minute = datetime_str[11:13]
    second = datetime_str[13:15]
    
    # Format with dashes and colons, add Z for UTC
    return f"{year}-{month}-{day}T{hour}:{minute}:{second}Z"

# Test
datetime_str = '20240430T002653'
result = convert_sentinel_datetime(datetime_str)
print(result)  # 2024-04-30T00:26:53Z

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [ ]:
def makedirs(data_download_dir, name, data_download_dir):
    # Create necessary directories
    os.makedirs("reproj", exist_ok=True)
    
    # Create data_download directory for caching
    data_download_dir = "data_download"
    os.makedirs(data_download_dir, exist_ok=True)
    
    # Create subdirectory structure to match S3 path
    s3_path_parts = name.split('/')
    local_subdir = os.path.join(data_download_dir, *s3_path_parts[:-1])
    os.makedirs(local_subdir, exist_ok=True)

    # Local path for the downloaded file (persistent storage)
    local_download_path = os.path.join(data_download_dir, name)
    
    return data_download_dir, local_subdir, local_download_path

In [ ]:
def convert_to_proper_CRS_and_cogify_chunked(name, cog_filename, cog_data_bucket, cog_data_prefix, 
                                            local_output_dir=None, chunk_config=None):
    """
    Convert a file to Cloud Optimized GeoTIFF with proper CRS using chunked processing.
    
    This function includes:
    - Chunked processing for memory efficiency
    - Download caching to avoid re-downloading files
    - CRS reprojection to EPSG:4326
    - COG validation before upload
    - Upload to S3
    - Smart nodata value handling based on data type
    - Memory monitoring and progress tracking
    """
    if chunk_config is None:
        chunk_config = CHUNK_CONFIG
    
    s3_key = f"{cog_data_prefix}/{cog_filename}"
    reproject_filename = f"reproj/{cog_filename}"

    #Make directories
    data_download_dir, local_subdir, local_download_path = makedirs(data_download_dir, name, data_download_dir)
    
    # Temporary file for processing
    temp_input_file = f"temp_{os.path.basename(name)}"
    
    # Memory monitoring
    if chunk_config.get('enable_memory_monitoring', True):
        initial_memory = get_memory_usage()
        print(f"   [MEMORY] Initial: {initial_memory:.1f} MB")

    try:
        import shutil
        
        # Check if file already exists locally
        if os.path.exists(local_download_path):
            print(f"   [CACHE HIT] Using cached file: {local_download_path}")
            shutil.copy(local_download_path, temp_input_file)
        else:
            # Download the file from S3
            print(f"   [DOWNLOAD] Downloading from S3...")
            s3_client.download_file(BUCKET, name, local_download_path)
            print(f"   [DOWNLOAD] ✅ Saved to cache")
            shutil.copy(local_download_path, temp_input_file)
        
        # Open source file and get metadata
        with rasterio.open(temp_input_file) as src:
            dst_crs = "EPSG:4326"
            
            # Check if reprojection is needed
            if src.crs and src.crs.to_string() == dst_crs:
                print(f"   [REPROJECT] Already in {dst_crs}, skipping reprojection")
                import shutil
                shutil.copy(temp_input_file, reproject_filename)
            else:
                print(f"   [REPROJECT] Converting to EPSG:4326 using chunked processing...")
                
                # Calculate transform for destination
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds
                )
                
                # Calculate optimal chunk size
                chunk_size = calculate_optimal_chunk_size(
                    width, height, src.count, src.dtypes[0],
                    memory_limit_mb=chunk_config.get('memory_limit_mb', 500)
                )
                
                # Prepare output kwargs
                kwargs = src.meta.copy()
                kwargs.update({
                    "driver": "GTiff",  # Use GTiff for intermediate file
                    "compress": "DEFLATE",
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height,
                    "tiled": True,
                    "blockxsize": min(chunk_size, width),
                    "blockysize": min(chunk_size, height)
                })
                
                # Create output file
                with rasterio.open(reproject_filename, "w", **kwargs) as dst:
                    # Calculate number of chunks
                    n_chunks_x = (width + chunk_size - 1) // chunk_size
                    n_chunks_y = (height + chunk_size - 1) // chunk_size
                    total_chunks = n_chunks_x * n_chunks_y
                    
                    print(f"   [CHUNKS] Processing {total_chunks} chunks ({n_chunks_x}x{n_chunks_y})")
                    
                    # Process each band
                    for band_idx in range(1, src.count + 1):
                        print(f"   [BAND {band_idx}/{src.count}] Processing...")
                        
                        # Use tqdm for progress tracking if enabled
                        if chunk_config.get('show_progress', True):
                            chunk_iterator = tqdm(
                                total=total_chunks,
                                desc=f"Band {band_idx}",
                                unit="chunks",
                                leave=False
                            )
                        else:
                            chunk_iterator = None
                        
                        # Process chunks
                        for y in range(0, height, chunk_size):
                            for x in range(0, width, chunk_size):
                                # Define window for this chunk
                                win_width = min(chunk_size, width - x)
                                win_height = min(chunk_size, height - y)
                                window = Window(x, y, win_width, win_height)
                                
                                # Create temporary arrays for chunk
                                chunk_data = np.zeros((win_height, win_width), dtype=src.dtypes[0])
                                
                                # Reproject chunk
                                reproject(
                                    source=rasterio.band(src, band_idx),
                                    destination=chunk_data,
                                    src_transform=src.transform,
                                    src_crs=src.crs,
                                    dst_transform=transform * rasterio.windows.transform(window, transform),
                                    dst_crs=dst_crs,
                                    resampling=Resampling.nearest,
                                    wrapdateline=True
                                )
                                
                                # Write chunk to output
                                dst.write(chunk_data, band_idx, window=window)
                                
                                # Update progress
                                if chunk_iterator:
                                    chunk_iterator.update(1)
                                
                                # Force garbage collection periodically
                                if (y // chunk_size * n_chunks_x + x // chunk_size) % 10 == 0:
                                    gc.collect()
                                    
                                    if chunk_config.get('enable_memory_monitoring', True):
                                        current_memory = get_memory_usage()
                                        if current_memory > initial_memory * 2:
                                            print(f"\n   [MEMORY] High usage: {current_memory:.1f} MB, forcing cleanup...")
                                            gc.collect()
                        
                        if chunk_iterator:
                            chunk_iterator.close()
        
        # COGify & upload
        print(f"   [COGIFY] Creating COG from reprojected file...")
        
        # Use rasterio to create COG
        with rasterio.open(reproject_filename) as src:
            # Smart nodata value handling based on data type
            print(f"   [NODATA] Data type: {src.dtypes[0]}")
            if src.dtypes[0] == 'uint8':
                nodata_value = 0
                print(f"   [NODATA] Using nodata value {nodata_value} for uint8 data")
            elif src.dtypes[0] == 'uint16':
                nodata_value = 0
                print(f"   [NODATA] Using nodata value {nodata_value} for uint16 data")
            else:
                nodata_value = -9999
                print(f"   [NODATA] Using nodata value {nodata_value} for {src.dtypes[0]} data")
            
            # Update profile for COG
            profile = src.profile.copy()
            profile.update(COG_PROFILE)
            profile['nodata'] = nodata_value
            
            with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
                tmp_name = tmp.name
                
                # Write COG using chunked approach
                with rasterio.open(tmp_name, 'w', **profile) as dst:
                    # Process in chunks to avoid memory issues
                    for band_idx in range(1, src.count + 1):
                        for y in range(0, src.height, chunk_size):
                            for x in range(0, src.width, chunk_size):
                                win_width = min(chunk_size, src.width - x)
                                win_height = min(chunk_size, src.height - y)
                                window = Window(x, y, win_width, win_height)
                                
                                data = src.read(band_idx, window=window)
                                dst.write(data, band_idx, window=window)
                
                # Validate COG
                print(f"   [VALIDATE] Checking COG validity...")
                is_valid_cog, validation_details = validate_cog(tmp_name)
                
                if is_valid_cog:
                    print(f"   [VALIDATE] ✅ Valid COG")
                else:
                    print(f"   [VALIDATE] ⚠️ COG validation warnings")
                    critical_errors = [e for e in validation_details.get('errors', []) if 'Invalid driver' in e]
                    if critical_errors:
                        raise ValueError(f"Critical COG validation failed")
                    if 'errors' in validation_details:
                        for error in validation_details['errors']:
                            print(f"      - {error}")
                    if 'warnings' in validation_details:
                        for warning in validation_details['warnings']:
                            print(f"      - {warning}")
                
                # Upload to S3
                print(f"   [UPLOAD] Uploading to S3...")
                s3_client.upload_file(
                    Filename=tmp_name,
                    Bucket=cog_data_bucket,
                    Key=s3_key
                )
                print(f"   [SUCCESS] ✅ Uploaded to s3://{cog_data_bucket}/{s3_key}")
                
                # Save locally if specified
                if local_output_dir:
                    os.makedirs(local_output_dir, exist_ok=True)
                    local_path = os.path.join(local_output_dir, cog_filename)
                    import shutil
                    shutil.copy(tmp_name, local_path)
        
        # Final memory report
        if chunk_config.get('enable_memory_monitoring', True):
            final_memory = get_memory_usage()
            print(f"   [MEMORY] Final: {final_memory:.1f} MB (Change: {final_memory - initial_memory:+.1f} MB)")
            
    except Exception as e:
        print(f"   [ERROR] Failed: {str(e)}")
        raise
            
    finally:
        # Clean up temporary files
        for temp_file in [temp_input_file, reproject_filename]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        if 'tmp_name' in locals() and os.path.exists(tmp_name):
            os.remove(tmp_name)
        
        # Force final garbage collection
        gc.collect()

print("✅ Chunked COG conversion function defined with memory-efficient processing")

In [ ]:
# Check current cache status using the imported function
check_cache_status()

# Process files

In [ ]:
# For simplicity, let's use python list comprehension to return the files
# We may need to rename them in different ways for different products
# We will do a similar process later

## NOTE --- We can actually use these objects since they have the same path as the s3 files. We will call them again later

cir = [f for f in keys if "cir" in f]
print(cir)
swir = [f for f in keys if "swir" in f]
print(swir)
true = [f for f in keys if "true" in f]
print(true)

In [ ]:
# Define filename creator functions for different file types

def create_cog_filename(f, EVENT_NAME):
    """Create COG filename for water mask files."""
    f2 = Path(f).stem
    f2
    f25 = f2.split('_')
    # print(f25)
    a='_'.join(f25[0:2])
    cog_filename=f'{EVENT_NAME}_{a}_{f25[-1]}_{f25[-2]}day.tif'
    # print(cog_filename)

    return cog_filename


# Test functions
print("Testing WM filename:")
test_wm = create_cog_filename('drcs_activations/202405_Flood_TX/sentinel2/cir/S2B_colorInfrared_20240505_merged.tif', EVENT_NAME)
print(f"  {test_wm}")

In [ ]:
# Process cir files with chunked processing
if cir:
    print("\n" + "="*50)
    print("🌊 Processing Water Mask Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function instead of the original
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    wm_results = process_file_batch(
        file_list=cir,
        s3_client=s3_client,
        config=config_cir,
        filename_creator_func=create_cog_filename,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, wm_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)

In [ ]:
# Process RGB files with chunked processing
if swir:
    print("\n" + "="*50)
    print("🎨 Processing RGB Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    rgb_results = process_file_batch(
        file_list=swir,
        s3_client=s3_client,
        config=config_swir,
        filename_creator_func=create_cog_filename,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, rgb_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)

In [ ]:
# Process water mask diff files with chunked processing
if true:
    print("\n" + "="*50)
    print("🔄 Processing Water Mask Diff Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    diff_results = process_file_batch(
        file_list=true,
        s3_client=s3_client,
        config=config_true,
        filename_creator_func=create_cog_filename,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, diff_results], ignore_index=True)

    # Print overall summary
    print_batch_summary(all_files_processed)

In [ ]:
# Display final results (for a single instance)
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.